## XGBOOST

In [ ]:
import os
import warnings
import json
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
import joblib
from datetime import datetime
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# Suppress warnings
warnings.filterwarnings('ignore')

# Define paths
data_path = os.path.join("..", "data", "train-test")
models_path = os.path.join("..", "models")

# Create directories if they don't exist
os.makedirs(data_path, exist_ok=True)
os.makedirs(models_path, exist_ok=True)

# Load preprocessed data
print("Loading preprocessed data...")
train_df = pd.read_csv(os.path.join(data_path, "train_set.csv"))
test_df = pd.read_csv(os.path.join(data_path, "test_set.csv"))

# Separate features and target
X_train = train_df.drop(columns=['has_diabetes'])
y_train = train_df['has_diabetes']
X_test = test_df.drop(columns=['has_diabetes'])
y_test = test_df['has_diabetes']

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Target distribution (train): {y_train.value_counts(normalize=True).map(lambda x: f'{x:.2%}')}")
print(f"Target distribution (test): {y_test.value_counts(normalize=True).map(lambda x: f'{x:.2%}')}")

# Wrapper for XGBoost booster to make it sklearn-compatible
class XGBoostWrapper:
    def __init__(self, booster):
        self.booster = booster

    def predict(self, X):
        dtest = xgb.DMatrix(X)
        proba = self.booster.predict(dtest)
        return (proba > 0.5).astype(int)

    def predict_proba(self, X):
        dtest = xgb.DMatrix(X)
        proba = self.booster.predict(dtest)
        return np.column_stack([1 - proba, proba])

# HYPERPARAMETER TUNING WITH OPTUNA
def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'eta': trial.suggest_float('eta', 0.01, 0.3, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 2.0),
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'seed': 42,
        'verbosity': 0,
        'nthread': 16
    }

    n_estimators = trial.suggest_int('n_estimators', 200, 1000)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_val, label=y_val)

        model = xgb.train(
            params,
            dtrain,
            num_boost_round=n_estimators,
            evals=[(dval, 'val')],
            early_stopping_rounds=50,
            verbose_eval=False
        )

        y_pred_proba = model.predict(dval)
        score = roc_auc_score(y_val, y_pred_proba)
        scores.append(score)

    return np.mean(scores)

print("\nStarting hyperparameter optimization with Optuna...")
study = optuna.create_study(direction='maximize', study_name='xgboost_diabetes')
study.optimize(objective, n_trials=10, show_progress_bar=True)

print(f"\nBest parameters found:")
best_params = study.best_params
for key, value in best_params.items():
    print(f"  {key}: {value}")
print(f"Best CV AUC: {study.best_value:.4f}")

# TRAIN FINAL MODEL
print("\nTraining final XGBoost model with best parameters...")

# Add fixed params
fixed_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'seed': 42,
    'verbosity': 0,
    'nthread': 16
}
train_params = {k: v for k, v in best_params.items() if k != 'n_estimators'}
train_params.update(fixed_params)

n_estimators = best_params['n_estimators']

# Create a held-out calibration set (20% of original train)
X_train_split, X_calib, y_train_split, y_calib = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

# Final train/val split for early stopping (20% of X_train_split)
X_train_final, X_val_final, y_train_final, y_val_final = train_test_split(
    X_train_split, y_train_split, test_size=0.2, stratify=y_train_split, random_state=42
)

# Train on combined train+val (i.e., X_train_split)
X_full_train = pd.concat([X_train_final, X_val_final])
y_full_train = pd.concat([y_train_final, y_val_final])

dtrain_full = xgb.DMatrix(X_full_train, label=y_full_train)
dval_final = xgb.DMatrix(X_val_final, label=y_val_final)

final_booster = xgb.train(
    train_params,
    dtrain_full,
    num_boost_round=n_estimators,
    evals=[(dval_final, 'val')],
    early_stopping_rounds=100,
    verbose_eval=False
)

print("Base model trained.")

# CALIBRATE ON HELD-OUT SET
print("\nCalibrating model for reliable probabilities...")
final_model = XGBoostWrapper(final_booster)
calibrated_model = CalibratedClassifierCV(final_model, method='isotonic', cv='prefit')
calibrated_model.fit(X_calib, y_calib)
print("Model calibrated.")

# PREDICTIONS
y_pred_proba = calibrated_model.predict_proba(X_test)[:, 1]
y_pred = calibrated_model.predict(X_test)

# BASIC EVALUATION
print("\nFinal Model Evaluation on Test Set:")
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC AUC:   {auc:.4f}")

# SAVE MODELS AND RESULTS
print("\nSaving models and results...")
joblib.dump(final_booster, os.path.join(models_path, "xgboost_diabetes_final.joblib"))
joblib.dump(calibrated_model, os.path.join(models_path, "xgboost_diabetes_calibrated.joblib"))
joblib.dump(study, os.path.join(models_path, "optuna_study_xgboost.joblib"))

results = {
    'best_params': best_params,
    'test_metrics': {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': auc
    },
    'timestamp': str(datetime.now())
}

with open(os.path.join(models_path, "model_results_xgboost.json"), 'w') as f:
    json.dump(results, f, indent=4, default=str)

print("Models and results saved successfully.")

Loading preprocessed data...


[I 2025-10-05 11:25:47,477] A new study created in memory with name: xgboost_diabetes


Train set: (291299, 14)
Test set: (72825, 14)
Target distribution (train): has_diabetes
1.0    50.00%
0.0    50.00%
Name: proportion, dtype: object
Target distribution (test): has_diabetes
0.0    50.00%
1.0    50.00%
Name: proportion, dtype: object

Starting hyperparameter optimization with Optuna...


Best trial: 0. Best value: 0.82458:  10%|█         | 1/10 [00:17<02:39, 17.67s/it]

[I 2025-10-05 11:26:05,143] Trial 0 finished with value: 0.824580092632074 and parameters: {'max_depth': 10, 'learning_rate': 0.14458493308325945, 'n_estimators': 664, 'subsample': 0.992732548076535, 'colsample_bytree': 0.8943979843195308, 'min_child_weight': 7, 'gamma': 0.3153793890172435, 'reg_alpha': 5.975552510199821, 'reg_lambda': 0.8897964866096353}. Best is trial 0 with value: 0.824580092632074.


...

Best trial: 3. Best value: 0.825751:  90%|█████████ | 9/10 [13:11<01:16, 76.41s/it] 

[I 2025-10-05 11:38:58,854] Trial 8 finished with value: 0.8241778728205421 and parameters: {'max_depth': 6, 'learning_rate': 0.22636109829056628, 'n_estimators': 894, 'subsample': 0.632606194094239, 'colsample_bytree': 0.9116997222057939, 'min_child_weight': 2, 'gamma': 1.1385055128215371, 'reg_alpha': 1.5316535463789227, 'reg_lambda': 1.3814810045835557}. Best is trial 3 with value: 0.8257511708963048.


Best trial: 3. Best value: 0.825751: 100%|██████████| 10/10 [14:11<00:00, 85.12s/it]


[I 2025-10-05 11:39:58,711] Trial 9 finished with value: 0.8252624000297205 and parameters: {'max_depth': 3, 'learning_rate': 0.1581117014048097, 'n_estimators': 329, 'subsample': 0.8722390842537684, 'colsample_bytree': 0.9180012501722793, 'min_child_weight': 3, 'gamma': 1.7928788247796712, 'reg_alpha': 2.5102942251607283, 'reg_lambda': 6.846542759874748}. Best is trial 3 with value: 0.8257511708963048.

Best parameters found:
  max_depth: 7
  learning_rate: 0.011132610743767583
  n_estimators: 981
  subsample: 0.6240361972765406
  colsample_bytree: 0.7400052461655051
  min_child_weight: 4
  gamma: 3.7354378331868423
  reg_alpha: 2.48159555072467
  reg_lambda: 3.282056172906085
Best CV AUC: 0.8258

Training final XGBoost model with best parameters...
Final model trained.

Calibrating model for reliable probabilities...


NotFittedError: This XGBWrapper instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

## CatBoost

In [ ]:
import os
import warnings
import json
import numpy as np
import pandas as pd
import catboost as cb
import optuna
import joblib
from datetime import datetime
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# Suppress warnings
warnings.filterwarnings('ignore')

# Define paths
data_path = os.path.join("..", "data", "train-test")
models_path = os.path.join("..", "models")

# Create directories if they don't exist
os.makedirs(data_path, exist_ok=True)
os.makedirs(models_path, exist_ok=True)

# Load preprocessed data
print("Loading preprocessed data...")
train_df = pd.read_csv(os.path.join(data_path, "train_set.csv"))
test_df = pd.read_csv(os.path.join(data_path, "test_set.csv"))

# Separate features and target
X_train = train_df.drop(columns=['has_diabetes'])
y_train = train_df['has_diabetes']
X_test = test_df.drop(columns=['has_diabetes'])
y_test = test_df['has_diabetes']

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Target distribution (train): {y_train.value_counts(normalize=True).map(lambda x: f'{x:.2%}')}")
print(f"Target distribution (test): {y_test.value_counts(normalize=True).map(lambda x: f'{x:.2%}')}")

# HYPERPARAMETER TUNING WITH OPTUNA
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 200, 1000),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 0.0, 1.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 2.0),
        'eval_metric': 'AUC',
        'loss_function': 'Logloss',
        'random_seed': 42,
        'verbose': False,
        'thread_count': 16
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = cb.CatBoostClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=(X_val, y_val),
            early_stopping_rounds=50,
            verbose=False
        )

        y_pred_proba = model.predict_proba(X_val)[:, 1]
        score = roc_auc_score(y_val, y_pred_proba)
        scores.append(score)

    return np.mean(scores)

print("\nStarting hyperparameter optimization with Optuna...")
study = optuna.create_study(direction='maximize', study_name='catboost_diabetes')
study.optimize(objective, n_trials=10, show_progress_bar=True)

print(f"\nBest parameters found:")
best_params = study.best_params
for key, value in best_params.items():
    print(f"  {key}: {value}")
print(f"Best CV AUC: {study.best_value:.4f}")

# TRAIN FINAL MODEL
print("\nTraining final CatBoost model with best parameters...")

# Add fixed params
best_params.update({
    'eval_metric': 'AUC',
    'loss_function': 'Logloss',
    'random_seed': 42,
    'verbose': False,
    'thread_count': 16
})

# Create a held-out calibration set (20% of original train)
X_train_split, X_calib, y_train_split, y_calib = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

# Final train/val split for early stopping (20% of X_train_split)
X_train_final, X_val_final, y_train_final, y_val_final = train_test_split(
    X_train_split, y_train_split, test_size=0.2, stratify=y_train_split, random_state=42
)

# Train on combined train+val (i.e., X_train_split)
X_full_train = pd.concat([X_train_final, X_val_final])
y_full_train = pd.concat([y_train_final, y_val_final])

final_model = cb.CatBoostClassifier(**best_params)
final_model.fit(
    X_full_train, y_full_train,
    eval_set=(X_val_final, y_val_final),
    early_stopping_rounds=100,
    verbose=False
)

print("Base model trained.")

# CALIBRATE ON HELD-OUT SET
print("\nCalibrating model for reliable probabilities...")
calibrated_model = CalibratedClassifierCV(final_model, method='isotonic', cv='prefit')
calibrated_model.fit(X_calib, y_calib)
print("Model calibrated.")

# PREDICTIONS
y_pred_proba = calibrated_model.predict_proba(X_test)[:, 1]
y_pred = calibrated_model.predict(X_test)

# BASIC EVALUATION
print("\nFinal Model Evaluation on Test Set:")
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC AUC:   {auc:.4f}")

# SAVE MODELS AND RESULTS
print("\nSaving models and results...")
joblib.dump(final_model, os.path.join(models_path, "catboost_diabetes_final.joblib"))
joblib.dump(calibrated_model, os.path.join(models_path, "catboost_diabetes_calibrated.joblib"))
joblib.dump(study, os.path.join(models_path, "optuna_study_catboost.joblib"))

results = {
    'best_params': best_params,
    'test_metrics': {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': auc
    },
    'timestamp': str(datetime.now())
}

with open(os.path.join(models_path, "model_results_catboost.json"), 'w') as f:
    json.dump(results, f, indent=4, default=str)

print("Models and results saved successfully.")

Loading preprocessed data...


[I 2025-10-05 10:15:02,025] A new study created in memory with name: catboost_diabetes


Train set: (291299, 14)
Test set: (72825, 14)
Target distribution (train): has_diabetes
1.0    50.00%
0.0    50.00%
Name: proportion, dtype: object
Target distribution (test): has_diabetes
0.0    50.00%
1.0    50.00%
Name: proportion, dtype: object

Starting hyperparameter optimization with Optuna...


Best trial: 0. Best value: 0.825647:  10%|█         | 1/10 [00:13<01:59, 13.33s/it]

[I 2025-10-05 10:15:15,357] Trial 0 finished with value: 0.8256467055641714 and parameters: {'iterations': 700, 'depth': 9, 'learning_rate': 0.1760948036992402, 'l2_leaf_reg': 3.8949806567072214, 'bagging_temperature': 0.9071771152935141, 'random_strength': 0.28458391585856624, 'border_count': 54, 'scale_pos_weight': 1.86925549149066}. Best is trial 0 with value: 0.8256467055641714.


...

[I 2025-10-05 10:19:14,996] Trial 7 finished with value: 0.8261272831867762 and parameters: {'iterations': 759, 'depth': 9, 'learning_rate': 0.024309097589102973, 'l2_leaf_reg': 6.1077256515377325, 'bagging_temperature': 0.062005702576228616, 'random_strength': 0.7359623293636001, 'border_count': 108, 'scale_pos_weight': 1.1135576927300503}. Best is trial 3 with value: 0.8261971918455362.


Best trial: 3. Best value: 0.826197:  90%|█████████ | 9/10 [04:46<00:35, 35.45s/it]

[I 2025-10-05 10:19:48,144] Trial 8 finished with value: 0.825766592532767 and parameters: {'iterations': 876, 'depth': 10, 'learning_rate': 0.04887928336449959, 'l2_leaf_reg': 1.9794454581623309, 'bagging_temperature': 0.9455486059212203, 'random_strength': 0.45393544167112654, 'border_count': 160, 'scale_pos_weight': 1.921347217156995}. Best is trial 3 with value: 0.8261971918455362.


Best trial: 3. Best value: 0.826197: 100%|██████████| 10/10 [04:56<00:00, 29.68s/it]


[I 2025-10-05 10:19:58,827] Trial 9 finished with value: 0.8254334252059466 and parameters: {'iterations': 420, 'depth': 9, 'learning_rate': 0.24399138499535014, 'l2_leaf_reg': 1.8408831562688723, 'bagging_temperature': 0.7541424980818997, 'random_strength': 0.55923626474565, 'border_count': 94, 'scale_pos_weight': 1.4004830287101395}. Best is trial 3 with value: 0.8261971918455362.

Best parameters found:
  iterations: 827
  depth: 7
  learning_rate: 0.03834610607919945
  l2_leaf_reg: 8.198307658939408
  bagging_temperature: 0.07173291381283886
  random_strength: 0.41886705307079597
  border_count: 117
  scale_pos_weight: 1.7913038689646004
Best CV AUC: 0.8262

Training final CatBoost model with best parameters...
Base model trained.

Calibrating model for reliable probabilities...
Model calibrated.

Final Model Evaluation on Test Set:
Accuracy:  0.7493
Precision: 0.7313
Recall:    0.7882
F1-Score:  0.7587
ROC AUC:   0.8244

Saving models and results...
Models and results saved succes

## Extra Trees

In [7]:
import os
import warnings
import json
import numpy as np
import pandas as pd
import optuna
import joblib
from datetime import datetime
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# Suppress warnings
warnings.filterwarnings('ignore')

# Define paths
data_path = os.path.join("..", "data", "train-test")
models_path = os.path.join("..", "models")
os.makedirs(data_path, exist_ok=True)
os.makedirs(models_path, exist_ok=True)

# Load data
print("Loading preprocessed data...")
train_df = pd.read_csv(os.path.join(data_path, "train_set.csv"))
test_df = pd.read_csv(os.path.join(data_path, "test_set.csv"))

X_train = train_df.drop(columns=['has_diabetes'])
y_train = train_df['has_diabetes']
X_test = test_df.drop(columns=['has_diabetes'])
y_test = test_df['has_diabetes']

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Target distribution (train): {y_train.value_counts(normalize=True).map(lambda x: f'{x:.2%}')}")
print(f"Target distribution (test): {y_test.value_counts(normalize=True).map(lambda x: f'{x:.2%}')}")

# HYPERPARAMETER TUNING WITH OPTUNA
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'class_weight': 'balanced',
        'n_jobs': 16,
        'random_state': 42
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = ExtraTreesClassifier(**params)
        model.fit(X_tr, y_tr)
        y_pred_proba = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, y_pred_proba))

    return np.mean(scores)

print("\nStarting hyperparameter optimization with Optuna...")
study = optuna.create_study(direction='maximize', study_name='extratrees_diabetes')
study.optimize(objective, n_trials=10, show_progress_bar=True)

print(f"\nBest parameters found:")
best_params = study.best_params
for key, value in best_params.items():
    print(f"  {key}: {value}")
print(f"Best CV AUC: {study.best_value:.4f}")

# TRAIN FINAL MODEL
print("\nTraining final Extra Trees model...")
best_params.update({
    'class_weight': 'balanced',
    'n_jobs': 16,
    'random_state': 42
})

final_model = ExtraTreesClassifier(**best_params)
final_model.fit(X_train, y_train)  # Train on full training set

# CALIBRATE MODEL
print("Calibrating model for reliable probabilities...")
calibrated_model = CalibratedClassifierCV(final_model, method='isotonic', cv=3)
calibrated_model.fit(X_train, y_train)

# PREDICTIONS
y_pred_proba = calibrated_model.predict_proba(X_test)[:, 1]
y_pred = calibrated_model.predict(X_test)

# BASIC EVALUATION
print("\nFinal Model Evaluation on Test Set:")
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC AUC:   {auc:.4f}")

# SAVE
print("\nSaving models and results...")
joblib.dump(final_model, os.path.join(models_path, "extratrees_diabetes_final.joblib"))
joblib.dump(calibrated_model, os.path.join(models_path, "extratrees_diabetes_calibrated.joblib"))
joblib.dump(study, os.path.join(models_path, "optuna_study_extratrees.joblib"))

results = {
    'best_params': best_params,
    'test_metrics': {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': auc
    },
    'timestamp': str(datetime.now())
}

with open(os.path.join(models_path, "model_results_extratrees.json"), 'w') as f:
    json.dump(results, f, indent=4, default=str)

print("Models and results saved successfully.")

Loading preprocessed data...


[I 2025-10-05 10:21:07,620] A new study created in memory with name: extratrees_diabetes


Train set: (291299, 14)
Test set: (72825, 14)
Target distribution (train): has_diabetes
1.0    50.00%
0.0    50.00%
Name: proportion, dtype: object
Target distribution (test): has_diabetes
0.0    50.00%
1.0    50.00%
Name: proportion, dtype: object

Starting hyperparameter optimization with Optuna...


Best trial: 0. Best value: 0.819996:  10%|█         | 1/10 [00:56<08:25, 56.19s/it]

[I 2025-10-05 10:22:03,808] Trial 0 finished with value: 0.8199964504234061 and parameters: {'n_estimators': 767, 'max_depth': 14, 'min_samples_split': 7, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.8199964504234061.


...

[I 2025-10-05 10:26:30,294] Trial 7 finished with value: 0.8215169953609911 and parameters: {'n_estimators': 930, 'max_depth': 19, 'min_samples_split': 9, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 7 with value: 0.8215169953609911.


Best trial: 7. Best value: 0.821517:  90%|█████████ | 9/10 [05:56<00:38, 38.39s/it]

[I 2025-10-05 10:27:03,876] Trial 8 finished with value: 0.8212289337361046 and parameters: {'n_estimators': 464, 'max_depth': 18, 'min_samples_split': 9, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 7 with value: 0.8215169953609911.


Best trial: 7. Best value: 0.821517: 100%|██████████| 10/10 [06:19<00:00, 37.97s/it]


[I 2025-10-05 10:27:27,325] Trial 9 finished with value: 0.8204652364500866 and parameters: {'n_estimators': 338, 'max_depth': 17, 'min_samples_split': 4, 'min_samples_leaf': 7, 'max_features': 'sqrt'}. Best is trial 7 with value: 0.8215169953609911.

Best parameters found:
  n_estimators: 930
  max_depth: 19
  min_samples_split: 9
  min_samples_leaf: 2
  max_features: sqrt
Best CV AUC: 0.8215

Training final Extra Trees model...
Calibrating model for reliable probabilities...

Final Model Evaluation on Test Set:
Accuracy:  0.7458
Precision: 0.7224
Recall:    0.7984
F1-Score:  0.7585
ROC AUC:   0.8205

Saving models and results...
Models and results saved successfully.


## Random Forest

In [8]:
import os
import warnings
import json
import numpy as np
import pandas as pd
import optuna
import joblib
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# Suppress warnings
warnings.filterwarnings('ignore')

# Define paths
data_path = os.path.join("..", "data",  "train-test")
models_path = os.path.join("..", "models")
os.makedirs(data_path, exist_ok=True)
os.makedirs(models_path, exist_ok=True)

# Load data
print("Loading preprocessed data...")
train_df = pd.read_csv(os.path.join(data_path, "train_set.csv"))
test_df = pd.read_csv(os.path.join(data_path, "test_set.csv"))

X_train = train_df.drop(columns=['has_diabetes'])
y_train = train_df['has_diabetes']
X_test = test_df.drop(columns=['has_diabetes'])
y_test = test_df['has_diabetes']

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Target distribution (train): {y_train.value_counts(normalize=True).map(lambda x: f'{x:.2%}')}")
print(f"Target distribution (test): {y_test.value_counts(normalize=True).map(lambda x: f'{x:.2%}')}")

# HYPERPARAMETER TUNING WITH OPTUNA
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'class_weight': 'balanced',
        'n_jobs': 16,
        'random_state': 42
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = RandomForestClassifier(**params)
        model.fit(X_tr, y_tr)
        y_pred_proba = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, y_pred_proba))

    return np.mean(scores)

print("\nStarting hyperparameter optimization with Optuna...")
study = optuna.create_study(direction='maximize', study_name='randomforest_diabetes')
study.optimize(objective, n_trials=10, show_progress_bar=True)

print(f"\nBest parameters found:")
best_params = study.best_params
for key, value in best_params.items():
    print(f"  {key}: {value}")
print(f"Best CV AUC: {study.best_value:.4f}")

# TRAIN FINAL MODEL ON FULL TRAINING SET
print("\nTraining final Random Forest model on full training set...")
best_params.update({
    'class_weight': 'balanced',
    'n_jobs': 16,
    'random_state': 42
})

final_model = RandomForestClassifier(**best_params)
final_model.fit(X_train, y_train)  #  Use full training data

# CALIBRATE MODEL
print("Calibrating model for reliable probabilities...")
calibrated_model = CalibratedClassifierCV(final_model, method='isotonic', cv=3)
calibrated_model.fit(X_train, y_train)

# PREDICTIONS
y_pred_proba = calibrated_model.predict_proba(X_test)[:, 1]
y_pred = calibrated_model.predict(X_test)

# BASIC EVALUATION
print("\nFinal Model Evaluation on Test Set:")
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC AUC:   {auc:.4f}")

# SAVE MODELS AND RESULTS
print("\nSaving models and results...")
joblib.dump(final_model, os.path.join(models_path, "randomforest_diabetes_final.joblib"))
joblib.dump(calibrated_model, os.path.join(models_path, "randomforest_diabetes_calibrated.joblib"))
joblib.dump(study, os.path.join(models_path, "optuna_study_randomforest.joblib"))

results = {
    'best_params': best_params,
    'test_metrics': {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': auc
    },
    'timestamp': str(datetime.now())
}

with open(os.path.join(models_path, "model_results_randomforest.json"), 'w') as f:
    json.dump(results, f, indent=4, default=str)

print("Models and results saved successfully.")

Loading preprocessed data...


[I 2025-10-05 10:28:45,750] A new study created in memory with name: randomforest_diabetes


Train set: (291299, 14)
Test set: (72825, 14)
Target distribution (train): has_diabetes
1.0    50.00%
0.0    50.00%
Name: proportion, dtype: object
Target distribution (test): has_diabetes
0.0    50.00%
1.0    50.00%
Name: proportion, dtype: object

Starting hyperparameter optimization with Optuna...


Best trial: 0. Best value: 0.823905:  10%|█         | 1/10 [00:20<03:02, 20.30s/it]

[I 2025-10-05 10:29:06,053] Trial 0 finished with value: 0.8239052202343811 and parameters: {'n_estimators': 213, 'max_depth': 14, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 0 with value: 0.8239052202343811.


...

[I 2025-10-05 10:33:47,522] Trial 7 finished with value: 0.8232201859185837 and parameters: {'n_estimators': 125, 'max_depth': 19, 'min_samples_split': 4, 'min_samples_leaf': 10, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.8240870675204295.


Best trial: 1. Best value: 0.824087:  90%|█████████ | 9/10 [05:17<00:25, 25.57s/it]

[I 2025-10-05 10:34:03,744] Trial 8 finished with value: 0.8229226831577259 and parameters: {'n_estimators': 156, 'max_depth': 20, 'min_samples_split': 9, 'min_samples_leaf': 9, 'max_features': 'log2'}. Best is trial 1 with value: 0.8240870675204295.


Best trial: 1. Best value: 0.824087: 100%|██████████| 10/10 [06:42<00:00, 40.20s/it]


[I 2025-10-05 10:35:27,779] Trial 9 finished with value: 0.8205792548316151 and parameters: {'n_estimators': 797, 'max_depth': 20, 'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.8240870675204295.

Best parameters found:
  n_estimators: 557
  max_depth: 13
  min_samples_split: 6
  min_samples_leaf: 10
  max_features: sqrt
Best CV AUC: 0.8241

Training final Random Forest model on full training set...
Calibrating model for reliable probabilities...

Final Model Evaluation on Test Set:
Accuracy:  0.7473
Precision: 0.7259
Recall:    0.7948
F1-Score:  0.7588
ROC AUC:   0.8231

Saving models and results...
Models and results saved successfully.
